In [ ]:
import pandas as pd
import requests
import datetime
import time
import numpy as np

# --- CONFIGURATION ---

# API Configuration
# REPLACE THIS with your actual API key
API_KEY = "YOUR_OPENWEATHERMAP_API_KEY" 

# Target Coordinates (Seixal area)
LAT = 38.625833
LON = -9.086561

# Target Coordinates (Lisbon area)
# LAT_LISBON = 38.7557335
# LON_LISBON = -9.1582073
    
# File Paths
INPUT_FILE = 'aggregated_data.csv'
OUTPUT_FILE = 'weather_data.csv'

# API Endpoint Template
BASE_URL = "https://api.openweathermap.org/data/3.0/onecall/timemachine"

In [ ]:
def iso_to_rounded_epoch(iso_str: str) -> int:
    """
    Converts ISO-8601 timestamp to a Unix epoch rounded to the nearest hour.
    """
    try:
        # Handle 'Z' if present for UTC
        dt = datetime.datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
        timestamp = dt.timestamp()
        
        # Round to nearest hour (3600 seconds)
        hours = timestamp / 3600
        rounded_timestamp = round(hours) * 3600
        return int(rounded_timestamp)
    except Exception as e:
        print(f"Error converting timestamp {iso_str}: {e}")
        return None

def get_historical_weather(epoch: int, lat: float, lon: float, api_key: str):
    """
    Fetches historical weather data for a specific epoch time.
    """
    params = {
        'lat': lat,
        'lon': lon,
        'dt': epoch,
        'appid': api_key,
        'units': 'metric'
    }
    
    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'data' in data and len(data['data']) > 0:
            current = data['data'][0]
            return {
                'temperature': current.get('temp'),
                'humidity': current.get('humidity'),
                'pressure': current.get('pressure')
            }
    except Exception as e:
        print(f"API Request failed for epoch {epoch}: {e}")
        
    return None

In [ ]:
# 1. Load the source data
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} rows from {INPUT_FILE}")
except FileNotFoundError:
    print(f"Error: File {INPUT_FILE} not found.")
    df = pd.DataFrame()

# 2. Filter for rows we want to match (e.g., household environment)
rows_to_process = df[df["environment"] == "household"]
print(f"Found {len(rows_to_process)} rows to process.")

# List to hold the extracted data
new_weather_data = []

# 3. Iterate and Fetch
for idx, row in rows_to_process.iterrows():
    timestamp_str = row["timestamp"]
    epoch = iso_to_rounded_epoch(timestamp_str)
    
    if epoch:
        print(f"Processing {timestamp_str}...", end=" ")
        
        weather = get_historical_weather(epoch, LAT, LON, API_KEY)
        
        if weather:
            # Create a simplified entry with ONLY timestamp and weather data
            entry = {
                "timestamp": timestamp_str,
                "temperature": weather['temperature'],
                "humidity": weather['humidity'],
                "pressure": weather['pressure']
            }
            new_weather_data.append(entry)
            print(f"Success: {weather['temperature']}°C")
        else:
            print("Failed to fetch.")
            
        # Optional: Sleep to respect API rate limits
        # time.sleep(0.2)
    else:
        print(f"Skipping invalid timestamp at index {idx}")

print("Processing complete.")

In [ ]:
if len(new_weather_data) > 0:
    # Create a clean DataFrame
    weather_df = pd.DataFrame(new_weather_data)
    
    # Save to the new specific file
    weather_df.to_csv(OUTPUT_FILE, index=False)
    
    print(f"Successfully created '{OUTPUT_FILE}' with {len(weather_df)} rows.")
    print(weather_df.head())
else:
    print("No data collected, file not created.")